# ARC Prize 2026 - schema-correct baseline

DEBUG-ONLY notebook for Kaggle editor runs. Do not submit this to the leaderboard. It loads the official JSON files, includes a lightweight evaluator/viewer for public splits, builds a simple heuristic baseline, and can write a debug output file.

Design goal: inspect schema and viewer only. The daily leaderboard submission must be reserved for a real GPU candidate.

In [ ]:
from pathlib import Path
from collections import Counter
import json
import os

import numpy as np

RUN_PUBLIC_EVAL = True
WRITE_SUBMISSION = False

KAGGLE_DATA_DIR = Path('/kaggle/input/arc-prize-2026-arc-agi-2')
LOCAL_DATA_DIR = Path('../data/raw')
DATA_DIR = KAGGLE_DATA_DIR if KAGGLE_DATA_DIR.exists() else LOCAL_DATA_DIR

TRAIN_CHALLENGES_PATH = DATA_DIR / 'arc-agi_training_challenges.json'
TRAIN_SOLUTIONS_PATH = DATA_DIR / 'arc-agi_training_solutions.json'
EVAL_CHALLENGES_PATH = DATA_DIR / 'arc-agi_evaluation_challenges.json'
EVAL_SOLUTIONS_PATH = DATA_DIR / 'arc-agi_evaluation_solutions.json'
TEST_CHALLENGES_PATH = DATA_DIR / 'arc-agi_test_challenges.json'
SAMPLE_SUBMISSION_PATH = DATA_DIR / 'sample_submission.json'

print('DATA_DIR:', DATA_DIR)
print('KAGGLE_IS_COMPETITION_RERUN:', os.getenv('KAGGLE_IS_COMPETITION_RERUN'))

In [ ]:
def load_json(path):
    with open(path, 'r') as f:
        return json.load(f)


def load_if_exists(path):
    return load_json(path) if path.exists() else None


train_challenges = load_if_exists(TRAIN_CHALLENGES_PATH)
train_solutions = load_if_exists(TRAIN_SOLUTIONS_PATH)
eval_challenges = load_if_exists(EVAL_CHALLENGES_PATH)
eval_solutions = load_if_exists(EVAL_SOLUTIONS_PATH)
test_challenges = load_json(TEST_CHALLENGES_PATH)
sample_submission = load_if_exists(SAMPLE_SUBMISSION_PATH)

print('train tasks:', 0 if train_challenges is None else len(train_challenges))
print('eval tasks:', 0 if eval_challenges is None else len(eval_challenges))
print('test tasks:', len(test_challenges))
if sample_submission is not None:
    print('sample submission tasks:', len(sample_submission))

## Submission scoring helpers

The official metric gives two attempts per test input. A test output scores 1 if either attempt exactly matches the hidden grid.

In [ ]:
def grid_to_tuple(grid):
    return tuple(tuple(int(x) for x in row) for row in grid)


def is_valid_grid(grid):
    if not isinstance(grid, list) or not grid:
        return False
    if not all(isinstance(row, list) and row for row in grid):
        return False
    width = len(grid[0])
    if not all(len(row) == width for row in grid):
        return False
    height = len(grid)
    if height > 30 or width > 30:
        return False
    return all(isinstance(v, int) and 0 <= v <= 9 for row in grid for v in row)


def validate_submission(challenges, submission):
    errors = []
    for task_id, task in challenges.items():
        if task_id not in submission:
            errors.append((task_id, 'missing task'))
            continue
        if len(submission[task_id]) != len(task['test']):
            errors.append((task_id, 'wrong number of test outputs'))
            continue
        for idx, attempts in enumerate(submission[task_id]):
            if sorted(attempts.keys()) != ['attempt_1', 'attempt_2']:
                errors.append((task_id, f'test {idx}: expected attempt_1 and attempt_2'))
                continue
            for name in ['attempt_1', 'attempt_2']:
                if not is_valid_grid(attempts[name]):
                    errors.append((task_id, f'test {idx}: invalid {name}'))
    return errors


def score_submission(challenges, solutions, submission):
    errors = validate_submission(challenges, submission)
    if errors:
        return {'score': None, 'hits': 0, 'total': 0, 'errors': errors[:20]}

    hits = 0
    total = 0
    solved_task_ids = []
    missed_task_ids = []
    for task_id, expected_list in solutions.items():
        task_hits = 0
        for idx, expected in enumerate(expected_list):
            attempts = submission[task_id][idx]
            expected_key = grid_to_tuple(expected)
            got = (
                grid_to_tuple(attempts['attempt_1']) == expected_key
                or grid_to_tuple(attempts['attempt_2']) == expected_key
            )
            hits += int(got)
            task_hits += int(got)
            total += 1
        if task_hits == len(expected_list):
            solved_task_ids.append(task_id)
        else:
            missed_task_ids.append(task_id)

    return {
        'score': hits / total if total else 0.0,
        'hits': hits,
        'total': total,
        'solved_tasks': len(solved_task_ids),
        'missed_tasks': len(missed_task_ids),
        'missed_task_ids_head': missed_task_ids[:20],
        'errors': [],
    }

## Viewer

Use this on Kaggle for visual debugging. It is intentionally lightweight and optional.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import colors

ARC_COLORS = [
    '#000000', '#0074D9', '#FF4136', '#2ECC40', '#FFDC00',
    '#AAAAAA', '#F012BE', '#FF851B', '#7FDBFF', '#870C25'
]
ARC_CMAP = colors.ListedColormap(ARC_COLORS)
ARC_NORM = colors.BoundaryNorm(list(range(11)), ARC_CMAP.N)


def draw_grid(ax, grid, title=''):
    arr = np.array(grid, dtype=int)
    ax.imshow(arr, cmap=ARC_CMAP, norm=ARC_NORM)
    ax.set_title(title, fontsize=9)
    ax.set_xticks(np.arange(-0.5, arr.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, arr.shape[0], 1), minor=True)
    ax.grid(which='minor', color='#555555', linewidth=0.5)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)


def show_task(task_id, challenges, solutions=None, submission=None):
    task = challenges[task_id]
    n_train = len(task['train'])
    n_test = len(task['test'])
    extra_cols = 1
    if solutions is not None and task_id in solutions:
        extra_cols += 1
    if submission is not None and task_id in submission:
        extra_cols += 2

    rows = max(n_train, n_test)
    cols = 2 + extra_cols
    fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3 * rows))
    axes = np.array(axes).reshape(rows, cols)
    for ax in axes.ravel():
        ax.axis('off')

    for i, pair in enumerate(task['train']):
        draw_grid(axes[i, 0], pair['input'], f'train {i} input')
        draw_grid(axes[i, 1], pair['output'], f'train {i} output')

    for i, case in enumerate(task['test']):
        col = 2
        draw_grid(axes[i, col], case['input'], f'test {i} input')
        col += 1
        if solutions is not None and task_id in solutions:
            draw_grid(axes[i, col], solutions[task_id][i], f'test {i} solution')
            col += 1
        if submission is not None and task_id in submission:
            draw_grid(axes[i, col], submission[task_id][i]['attempt_1'], f'test {i} attempt 1')
            draw_grid(axes[i, col + 1], submission[task_id][i]['attempt_2'], f'test {i} attempt 2')

    fig.suptitle(task_id)
    plt.tight_layout()
    plt.show()

## Lightweight baseline

This baseline is intentionally simple: detect a small set of exact transformations from the demonstrations, generate candidates, deduplicate, and keep two attempts. It is debug-only by default and must not consume the daily leaderboard submission.

In [ ]:
def to_array(grid):
    return np.array(grid, dtype=int)


def to_grid(arr):
    return np.asarray(arr, dtype=int).tolist()


def same_grid(a, b):
    return grid_to_tuple(a) == grid_to_tuple(b)


def most_common_color(grid):
    counts = Counter(int(v) for row in grid for v in row)
    return counts.most_common(1)[0][0]


def solid_grid(height, width, color):
    return [[int(color) for _ in range(width)] for _ in range(height)]


def crop_non_background(grid):
    arr = to_array(grid)
    bg = most_common_color(grid)
    mask = arr != bg
    if not mask.any():
        return grid
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    return to_grid(arr[rows.min():rows.max() + 1, cols.min():cols.max() + 1])


TRANSFORMS = [
    ('identity', lambda a: a),
    ('rot90', lambda a: np.rot90(a, 1)),
    ('rot180', lambda a: np.rot90(a, 2)),
    ('rot270', lambda a: np.rot90(a, 3)),
    ('flip_lr', lambda a: np.fliplr(a)),
    ('flip_ud', lambda a: np.flipud(a)),
    ('transpose', lambda a: a.T),
    ('anti_transpose', lambda a: np.fliplr(np.flipud(a).T)),
]


def learn_color_map(train_pairs):
    mapping = {}
    for pair in train_pairs:
        inp = pair['input']
        out = pair['output']
        if np.array(inp).shape != np.array(out).shape:
            return None
        for src, dst in zip(np.ravel(inp), np.ravel(out)):
            src = int(src)
            dst = int(dst)
            if src in mapping and mapping[src] != dst:
                return None
            mapping[src] = dst
    return mapping if mapping else None


def apply_color_map(grid, mapping):
    return [[int(mapping.get(int(v), int(v))) for v in row] for row in grid]


def detect_repeat_scale(train_pairs):
    scale = None
    for pair in train_pairs:
        inp = to_array(pair['input'])
        out = to_array(pair['output'])
        if out.shape[0] % inp.shape[0] or out.shape[1] % inp.shape[1]:
            return None
        row_scale = out.shape[0] // inp.shape[0]
        col_scale = out.shape[1] // inp.shape[1]
        if row_scale < 1 or col_scale < 1:
            return None
        expanded = np.kron(inp, np.ones((row_scale, col_scale), dtype=int))
        if not np.array_equal(expanded, out):
            return None
        current = (row_scale, col_scale)
        if scale is not None and scale != current:
            return None
        scale = current
    return scale

In [ ]:
def build_solvers(task):
    train_pairs = task['train']
    solvers = []

    # Exact static output, useful for constant-pattern tasks.
    first_output = train_pairs[0]['output']
    if all(same_grid(pair['output'], first_output) for pair in train_pairs):
        solvers.append(('static_output', lambda grid, out=first_output: out))

    # Global geometric transforms.
    for name, fn in TRANSFORMS:
        ok = True
        for pair in train_pairs:
            try:
                pred = to_grid(fn(to_array(pair['input'])))
            except Exception:
                ok = False
                break
            if not same_grid(pred, pair['output']):
                ok = False
                break
        if ok:
            solvers.append((name, lambda grid, fn=fn: to_grid(fn(to_array(grid)))))

    # Consistent color substitution with same shape.
    mapping = learn_color_map(train_pairs)
    if mapping is not None:
        solvers.append(('color_map', lambda grid, mapping=mapping: apply_color_map(grid, mapping)))

    # Crop non-background bounding box.
    if all(same_grid(crop_non_background(pair['input']), pair['output']) for pair in train_pairs):
        solvers.append(('crop_non_background', crop_non_background))

    # Pixel repeat / nearest-neighbor expansion.
    scale = detect_repeat_scale(train_pairs)
    if scale is not None:
        solvers.append(('repeat_scale', lambda grid, scale=scale: to_grid(np.kron(to_array(grid), np.ones(scale, dtype=int)))))

    return solvers


def add_candidate(candidates, name, grid):
    if not is_valid_grid(grid):
        return
    key = grid_to_tuple(grid)
    if key not in {item[2] for item in candidates}:
        candidates.append((name, grid, key))


def generate_candidates(task, test_input):
    candidates = []
    for name, solver in build_solvers(task):
        try:
            add_candidate(candidates, name, solver(test_input))
        except Exception:
            pass

    arr = to_array(test_input)
    height, width = arr.shape
    bg = most_common_color(test_input)

    # Stable fallbacks. These are weak but keep the schema valid.
    add_candidate(candidates, 'identity_fallback', test_input)
    add_candidate(candidates, 'background_like_fallback', solid_grid(height, width, bg))
    add_candidate(candidates, 'zero_1x1_fallback', [[0]])

    return candidates


def attempts_for_input(task, test_input):
    candidates = generate_candidates(task, test_input)
    while len(candidates) < 2:
        add_candidate(candidates, f'pad_{len(candidates)}', [[len(candidates) % 10]])
    return {
        'attempt_1': candidates[0][1],
        'attempt_2': candidates[1][1],
    }


def make_submission(challenges):
    submission = {}
    for task_id, task in challenges.items():
        submission[task_id] = [attempts_for_input(task, case['input']) for case in task['test']]
    return submission

## Public validation

This cell scores the baseline on the public evaluation split when solutions are available. It is safe to leave in the Kaggle notebook; it does not use hidden test labels.

In [ ]:
if RUN_PUBLIC_EVAL and eval_challenges is not None and eval_solutions is not None:
    eval_submission = make_submission(eval_challenges)
    eval_result = score_submission(eval_challenges, eval_solutions, eval_submission)
    print(json.dumps(eval_result, indent=2))
else:
    eval_submission = None
    eval_result = None
    print('Public eval skipped.')

Optional visual check. Uncomment one missed task after public validation to inspect the prediction.

In [ ]:
# if eval_result and eval_result['missed_task_ids_head']:
#     task_id = eval_result['missed_task_ids_head'][0]
#     show_task(task_id, eval_challenges, eval_solutions, eval_submission)

## Write debug output

This debug notebook intentionally writes `debug_submission.json` by default. Do not use daily submission quota on this baseline.

In [ ]:
submission = make_submission(test_challenges)
submission_errors = validate_submission(test_challenges, submission)
print('submission tasks:', len(submission))
print('validation errors:', len(submission_errors))
if submission_errors:
    print(submission_errors[:20])
    raise ValueError('Invalid submission schema')

if WRITE_SUBMISSION:
    with open('debug_submission.json', 'w') as f:
        json.dump(submission, f)
    print('Wrote debug_submission.json; flip WRITE_SUBMISSION only for a real candidate')